# Explainability Analysis

Creates feature-importance artifacts for the dashboard. This notebook uses the same leakage-cleaned feature set as the Random Forest and XGBoost notebooks.

Run this after `Random_Forest_Model.ipynb` and `XGBoost_Model.ipynb` have saved their models.

## 1. Setup

In [1]:
from modeling_utils import (
    LEAKAGE_PRONE_COLS,
    MODEL_DIR,
    load_model_ready,
    load_saved_model,
    remove_leakage_prone_features,
    save_feature_columns,
    temporal_split,
    write_permutation_importance,
    write_shap_summary,
    write_tree_importance,
)

## 2. Load Clean Feature Set and Models

In [2]:
df, feature_cols = load_model_ready()
clean_feature_cols = remove_leakage_prone_features(feature_cols)
save_feature_columns(clean_feature_cols)
x_train, x_test, y_train, y_test = temporal_split(df, clean_feature_cols)

model_slugs = [
    slug for slug in ["random_forest", "xgboost"]
    if (MODEL_DIR / f"{slug}.joblib").exists()
]

print(f"Models found: {model_slugs}")
print(f"Removed leakage-prone columns: {sorted(LEAKAGE_PRONE_COLS)}")
print(f"Explanation feature count: {len(clean_feature_cols):,}")

Models found: ['random_forest', 'xgboost']
Removed leakage-prone columns: ['enrollment_actual', 'log_enrollment', 'trial_duration_days']
Explanation feature count: 144


## 3. Tree-Based Feature Importance

In [3]:
for slug in model_slugs:
    model = load_saved_model(slug)
    path = write_tree_importance(slug, model, clean_feature_cols)
    print(f"{slug}: {path}")

random_forest: artifacts\explainability\random_forest_feature_importance.csv
xgboost: artifacts\explainability\xgboost_feature_importance.csv


## 4. Permutation Importance

In [4]:
for slug in model_slugs:
    model = load_saved_model(slug)
    path = write_permutation_importance(slug, model, x_test, y_test, clean_feature_cols, sample_size=1500)
    print(f"{slug}: {path}")

random_forest: artifacts\explainability\random_forest_permutation_importance.csv
xgboost: artifacts\explainability\xgboost_permutation_importance.csv


## 5. SHAP Summary

In [5]:
for slug in model_slugs:
    model = load_saved_model(slug)
    path = write_shap_summary(slug, model, x_test, clean_feature_cols, sample_size=500)
    print(f"{slug}: {path}")

c:\Users\andre\245-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


random_forest: artifacts\explainability\random_forest_shap_summary.csv
xgboost: artifacts\explainability\xgboost_shap_summary.csv


## Notes

Feature importance ranks split gain/impurity from the tree model. Permutation importance measures macro-F1 drop when a feature is shuffled. SHAP gives the average absolute contribution of each feature across a sample of test rows.